# BRNS Result Comparison: Version A vs. Version B

This notebook compares `.dat` output files from two freely selectable result folders.
Use it to compare Maple vs. Python, or any two variants of a reaction network.

**Data format:** Column 0 = concentration/rate, Column 1 = depth

- **Version A:** solid blue line
- **Version B:** dashed orange line

For publication, prefer repository-relative paths over user-specific absolute paths.
The workflow is intentionally simple: configure two paths, load shared datasets, then plot overlays per time snapshot.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from notebooks.utils.helper import collect_common_dat_datasets, split_snapshots

# ============================================================
# Configuration: simple notebook-relative paths
# ============================================================
COMPARISON_NAME = 'equilibrium'
#COMPARISON_NAME = 'switches'
VERSION_A_LABEL = 'BRNS 2.0 (Maple)'
VERSION_B_LABEL = 'BRNS 3.0 (Python)'

# Notebook is in: BRNSPackage/notebooks
# Build data is in: BRNSPackage/build/<example>/results/...
BUILD_DIR = (Path('..') / 'BRNSPackage' / 'build').resolve()

# Example paths (adjust as needed or give full path as f-String):
VERSION_A_DIR = BUILD_DIR / COMPARISON_NAME / 'results' / 'reference'
VERSION_B_DIR = BUILD_DIR / COMPARISON_NAME / 'results' / 'python'

print('Comparison     :', COMPARISON_NAME)
print(f'{VERSION_A_LABEL:12}:', VERSION_A_DIR)
print(f'{VERSION_B_LABEL:12}:', VERSION_B_DIR)
print('A exists       :', VERSION_A_DIR.exists())
print('B exists       :', VERSION_B_DIR.exists())

datasets, common, only_a, only_b = collect_common_dat_datasets(VERSION_A_DIR, VERSION_B_DIR)

print("Build dir:", BUILD_DIR)
print('Common .dat files:', len(common))
if only_a:
    print(f'Only in {VERSION_A_LABEL}:', only_a)
if only_b:
    print(f'Only in {VERSION_B_LABEL}:', only_b)
print('Loaded:', len(datasets), 'files')

## Comparison Plots

Each file is shown as one figure with one subplot per detected time snapshot.
If both curves overlap, the two versions are numerically consistent for that dataset.

In [ ]:
# ============================================================
# Per file: one figure with ONE column (one subplot per time step)
# Time-step detection: a new step starts when depth resets
# (i.e., when x is smaller than in the previous row)
# ============================================================

for fname, d in sorted(datasets.items()):
    arr_a = d['ref']
    arr_b = d['py']

    snaps_a = split_snapshots(arr_a)
    snaps_b = split_snapshots(arr_b)
    n_snaps = max(len(snaps_a), len(snaps_b))

    if n_snaps == 0:
        print(f"⚠ {fname}: no data")
        continue

    print(f"\n=== {fname} ===")
    print(f"Snapshots {VERSION_A_LABEL}: {len(snaps_a)} | {VERSION_B_LABEL}: {len(snaps_b)}")

    fig, axes = plt.subplots(
        n_snaps,
        1,
        figsize=(10, max(3.8 * n_snaps, 4.5)),
        squeeze=False
    )
    axes = axes.flatten()

    for i in range(n_snaps):
        ax = axes[i]

        # Version A
        if i < len(snaps_a):
            snap_a = snaps_a[i]
            if snap_a.shape[1] >= 2:
                x_a, y_a = snap_a[:, 1], snap_a[:, 0]
            else:
                x_a, y_a = np.arange(len(snap_a)), snap_a[:, 0]
            ax.plot(x_a, y_a, '-', color='steelblue', linewidth=2.2, label=VERSION_A_LABEL)

        # Version B
        if i < len(snaps_b):
            snap_b = snaps_b[i]
            if snap_b.shape[1] >= 2:
                x_b, y_b = snap_b[:, 1], snap_b[:, 0]
            else:
                x_b, y_b = np.arange(len(snap_b)), snap_b[:, 0]
            ax.plot(x_b, y_b, '--', color='orange', linewidth=1.8, alpha=0.95, label=VERSION_B_LABEL)

        ax.set_title(f"{fname} | Time step {i+1}", fontsize=10, fontweight='bold')
        ax.set_xlabel('Depth', fontsize=9)
        ax.set_ylabel('Concentration / Rate', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
        ax.legend(fontsize=8, loc='best')

    fig.suptitle(f"Result comparison per time step: {fname}  [{COMPARISON_NAME}]", fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0.02, 1, 0.97])
    plt.show()

In [ ]:
# ============================================================
# Publication figure for TWO selected output files
# Last time step only (one panel per selected file)
# ============================================================
# Usage: set exactly two files in SELECTED_OUTPUT_FILES (name or full path),
# then run this cell after the data-loading cell.

from pathlib import Path
import matplotlib as mpl

SELECTED_OUTPUT_FILES = [
    'tol_h_l_B3.dat',
    'xrate37.dat',
]

SELECTED_OUTPUT_FILES = [
    'ch2o.dat',
    'fes.dat',
]


if 'datasets' not in globals():
    raise RuntimeError('Please run the data-loading cell first (datasets missing).')
if len(SELECTED_OUTPUT_FILES) != 2:
    raise ValueError('Please provide exactly two files in SELECTED_OUTPUT_FILES.')

mpl.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Serif',
    'font.size': 10,
    'axes.titlesize': 10,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 0.8,
    'lines.linewidth': 2.0,
})

def _find_dataset_key(entry, keys):
    s = str(entry)
    if s in keys:
        return s
    name = Path(s).name
    for k in keys:
        if Path(k).name == name:
            return k
    return None

def _xy(snap):
    if snap.shape[1] >= 2:
        return snap[:, 1], snap[:, 0]
    return np.arange(len(snap)), snap[:, 0]

rows = []
for entry in SELECTED_OUTPUT_FILES:
    key = _find_dataset_key(entry, datasets.keys())
    if key is None:
        raise FileNotFoundError(f"Dataset not found for '{entry}'.")

    d = datasets[key]
    snaps_a = split_snapshots(d['ref'])
    snaps_b = split_snapshots(d['py'])

    if len(snaps_a) == 0 and len(snaps_b) == 0:
        raise RuntimeError(f"No snapshots detected for '{key}'.")

    last_a = snaps_a[-1] if len(snaps_a) > 0 else None
    last_b = snaps_b[-1] if len(snaps_b) > 0 else None
    last_idx = max(len(snaps_a), len(snaps_b))
    rows.append((entry, key, last_a, last_b, last_idx, len(snaps_a), len(snaps_b)))

fig, axes = plt.subplots(
    2, 1, figsize=(8.0, 8.5), squeeze=False, constrained_layout=True
)
axes = axes.flatten()

for r, (entry, key, last_a, last_b, last_idx, n_a, n_b) in enumerate(rows):
    ax = axes[r]

    if last_a is not None:
        x_a, y_a = _xy(last_a)
        ax.plot(x_a, y_a, '-', color='steelblue', label=VERSION_A_LABEL)

    if last_b is not None:
        x_b, y_b = _xy(last_b)
        ax.plot(x_b, y_b, '--', color='darkorange', label=VERSION_B_LABEL)

    ax.grid(True, alpha=0.25)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax.set_xlabel('Depth')
    ax.set_ylabel('Concentration / Rate')
    ax.set_title(f"{Path(key).name} | Last time step (index {last_idx})")
    ax.legend(loc='best', frameon=False)

print('Selected files (last step only):')
for entry, key, _, _, last_idx, n_a, n_b in rows:
    print(' - input       :', entry)
    print('   key         :', key)
    print('   snapshots   :', f"{VERSION_A_LABEL}={n_a}, {VERSION_B_LABEL}={n_b}")
    print('   plotted step:', last_idx)

plt.savefig(BUILD_DIR / COMPARISON_NAME / 'results' / 'comparison.pdf', dpi=300)
plt.show()